# M2: exploitation probability compared with EPSS (#21, US13)

**Goal (criteria 1 and 2 of #21).** Predict, at the publication of a CVE, the probability that it is exploited, and compare this benchmark honestly with EPSS. M2 does not replace EPSS: EPSS stays an input of the priority score (criterion 4).

**Decisions** (team proposal validated with the supervisor, Product Owner; plan posted on #21):

| Topic | Choice |
|---|---|
| Label | Positive = added to the CISA KEV catalog **within 180 days** of publication (an addition before publication counts). Only CVEs whose 180 days are over at the KEV snapshot are used |
| Data start | CVEs published from **1 Jan 2022**: the KEV catalog started in Nov 2021 with a bulk addition of old CVEs, which would distort 2021 |
| Time split | Train 2022-01 → 2024-06, validation 2024-07 → 2024-12, **gap** 2025-01 → 2025-06 (their labels were not known on 1 Jul 2025), test 2025-07 → snapshot − 180 days |
| EPSS of the same date | EPSS of the **Sunday after publication** (dated weekly files), compared on the same CVEs |
| Metrics | PR-AUC (average precision), precision@k (k = 50, 100, 500), ROC-AUC; baselines: CNA CVSS score alone, random |

In [1]:
import re
import sys
import urllib.request
from datetime import date, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import psycopg
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import average_precision_score, roc_auc_score

def find_repo_root(start=Path.cwd()):
    for folder in [start, *start.parents]:
        if (folder / "docker-compose.yml").exists():
            return folder
    raise FileNotFoundError("Run the notebook from inside the threatlens repository")

def read_env(path):
    values = {}
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            key, value = line.split("=", 1)
            values[key.strip()] = value.strip()
    return values

ROOT = find_repo_root()
env = read_env(ROOT / ".env")
conn = psycopg.connect(host="localhost", port=int(env.get("POSTGRES_PORT", 5432)),
                       dbname=env["POSTGRES_DB"], user=env["POSTGRES_USER"], password=env["POSTGRES_PASSWORD"])

def query(sql, params=None):
    with conn.cursor() as cursor:
        cursor.execute(sql, params)
        return pd.DataFrame(cursor.fetchall(), columns=[column.name for column in cursor.description])

RANDOM_STATE = 42
HORIZON = 180
TRAIN_FROM, VALID_FROM, GAP_FROM, TEST_FROM = date(2022, 1, 1), date(2024, 7, 1), date(2025, 1, 1), date(2025, 7, 1)
pd.set_option("display.max_colwidth", 80)
print("Connected to", env["POSTGRES_DB"])

Connected to threatlens


## 1. Label and time split

The KEV snapshot is the latest `date_added` loaded. A CVE published less than 180 days before it has not had the time to enter KEV: it is left out (right censoring).

In [2]:
snapshot = query("SELECT max(date_added) AS d FROM cti_keventry")["d"].iloc[0]
last_published = snapshot - timedelta(days=HORIZON)

cves = query("""
    SELECT v.id, v.cve_id, (v.published_at AT TIME ZONE 'UTC')::date AS published,
           v.assigner, v.title, v.description, k.date_added AS kev_added
    FROM cti_vulnerability v
    LEFT JOIN cti_keventry k ON k.vulnerability_id = v.id
    WHERE v.published_at >= %(start)s
      AND (v.published_at AT TIME ZONE 'UTC')::date <= %(last)s
""", {"start": TRAIN_FROM, "last": last_published})
cves["published"] = pd.to_datetime(cves["published"])
cves["kev_added"] = pd.to_datetime(cves["kev_added"])
cves["label"] = (cves["kev_added"].notna()
                 & (cves["kev_added"] <= cves["published"] + pd.Timedelta(days=HORIZON))).astype(int)
cves["part"] = np.select(
    [cves["published"] < pd.Timestamp(VALID_FROM), cves["published"] < pd.Timestamp(GAP_FROM),
     cves["published"] < pd.Timestamp(TEST_FROM)],
    ["1 train", "2 validation", "3 gap (not used)"], "4 test")
cves = cves.set_index("id")

split_summary = cves.groupby("part").agg(cves=("label", "size"), positives=("label", "sum"),
                                         first_day=("published", "min"), last_day=("published", "max"))
split_summary["positive %"] = (100 * split_summary["positives"] / split_summary["cves"]).round(2)
print("KEV snapshot:", snapshot, "| last publication date used:", last_published)
split_summary

KEV snapshot: 2026-11-20 | last publication date used: 2026-05-24


,cves,positives,first_day,last_day,positive %
part,,,,,
1 train,74195,287,2022-01-01,2024-06-30,0.39
2 validation,19575,78,2024-07-01,2024-12-31,0.40
3 gap (not used),23642,96,2025-01-01,2025-06-30,0.41
4 test,50341,154,2025-07-01,2026-05-24,0.31


## 2. Features frozen at publication, and the leakage check (criterion 1)

| Used (all from the CNA record, published with the CVE) | Excluded, and why |
|---|---|
| CNA CVSS score and vector (newest version: 3.1, 3.0, 4.0, then 2.0) | NVD and CISA ADP scores: added after publication, no date per score; the CISA ADP SSVC *Exploitation* field is almost the label (FR-24) |
| CWEs named by the CNA | CWEs named by the NVD or CISA ADP: added later |
| Affected vendors and number of products | Current `epss_score` / `epss_percentile`: today's EPSS already knows the exploitation |
| CNA (`assigner`) | KEV fields, ThreatFox, alerts, `nvd_status`, effective severity, M1 prediction |
| Title and description, **with exploitation phrases masked** | `source_updated_at` and any date after publication |

The description stored is the latest version of the record: some CNAs add "exploited in the wild" after the KEV entry. The next cell counts these phrases and masks them.

In [3]:
EXPLOITATION_PHRASES = re.compile(
    r"exploited in the wild|actively exploited|active exploitation|being exploited|been exploited"
    r"|known to be exploited|known exploited|under active attack|targeted attacks?|in the wild"
    r"|\bkev\b|\bcisa\b", re.IGNORECASE)

cves["text"] = (cves["title"].fillna("") + " " + cves["description"].fillna("")).str.strip()
cves["has_phrase"] = cves["text"].str.contains(EXPLOITATION_PHRASES)
phrase_check = cves.groupby("label")["has_phrase"].agg(cves="size", with_phrase="sum")
phrase_check["with phrase %"] = (100 * phrase_check["with_phrase"] / phrase_check["cves"]).round(2)
cves["text"] = cves["text"].str.replace(EXPLOITATION_PHRASES, " ", regex=True)
print("Exploitation phrases before masking (label 1 = exploited within 180 days):")
phrase_check

Exploitation phrases before masking (label 1 = exploited within 180 days):


,cves,with_phrase,with phrase %
label,,,
0,167138,297,0.18
1,615,71,11.54


In [4]:
cvss = query("""
    SELECT vulnerability_id AS id, version, vector, base_score
    FROM cti_cvssmetric WHERE provider = 'CNA'
""")
version_order = {"3.1": 0, "3.0": 1, "4.0": 2, "2.0": 3}
cvss["order"] = cvss["version"].map(version_order).fillna(9)
cvss = cvss.sort_values(["id", "order"]).drop_duplicates("id").set_index("id")

def vector_parts(vector):
    parts = {}
    for item in str(vector or "").split("/"):
        if ":" in item and not item.startswith("CVSS"):
            key, value = item.split(":", 1)
            parts[key] = value
    return parts

parts = cvss["vector"].map(vector_parts)
cves["cna_score"] = cvss["base_score"].reindex(cves.index).astype(float)
cves["cna_version"] = cvss["version"].reindex(cves.index)
for name, keys in {"AV": ["AV"], "AC": ["AC"], "PR": ["PR", "Au"], "UI": ["UI"],
                   "C": ["C", "VC"], "I": ["I", "VI"], "A": ["A", "VA"]}.items():
    values = parts.map(lambda p: next((p[k] for k in keys if k in p), None))
    cves["cvss_" + name] = values.reindex(cves.index)

cwe = query("""
    SELECT vw.vulnerability_id AS id, w.cwe_id
    FROM cti_vulnerabilityweakness vw JOIN cti_weakness w ON w.id = vw.weakness_id
    WHERE vw.provider = 'CNA'
""")
cwe = cwe[cwe["id"].isin(cves.index)]
products = query("""
    SELECT ap.vulnerability_id AS id, p.vendor
    FROM cti_affectedproduct ap JOIN cti_product p ON p.id = ap.product_id
""")
products = products[products["id"].isin(cves.index)]
cves["n_products"] = products.groupby("id").size().reindex(cves.index).fillna(0)
cves["n_vendors"] = products.groupby("id")["vendor"].nunique().reindex(cves.index).fillna(0)
cves["n_cwes"] = cwe.groupby("id").size().reindex(cves.index).fillna(0)
cves["text_length"] = cves["text"].str.len()
print("CVEs with a CNA CVSS score:", f"{cves['cna_score'].notna().mean():.1%}",
      "| with a CNA CWE:", f"{(cves['n_cwes'] > 0).mean():.1%}",
      "| with an affected product:", f"{(cves['n_products'] > 0).mean():.1%}")

CVEs with a CNA CVSS score: 67.9% | with a CNA CWE: 64.6% | with an affected product: 78.5%


## 3. Model (FR-24)

Gradient boosting (`HistGradientBoostingClassifier`) on the structured features, plus the text reduced by TF-IDF → TruncatedSVD (100 dimensions). Every vocabulary, top list and reduction is learnt on the training part only. The rare positives (about 0.4 %) are weighted. A few settings are compared on the validation part, then the chosen one is refit on train + validation (labels all known before 1 Jul 2025) and measured once on the test part.

In [5]:
CATEGORICAL = ["assigner", "cna_version", "cvss_AV", "cvss_AC", "cvss_PR", "cvss_UI", "cvss_C", "cvss_I", "cvss_A"]
NUMERIC = ["cna_score", "n_products", "n_vendors", "n_cwes", "text_length"]

class Features:
    """Learns the vocabularies on a training part, then turns any part into a matrix."""

    def __init__(self, train, top_vendors=100, top_cwes=50, top_categories=50, svd_dims=100):
        self.categories = {c: train[c].dropna().value_counts().index[:top_categories].tolist() for c in CATEGORICAL}
        self.vendors = products[products["id"].isin(train.index)]["vendor"].value_counts().index[:top_vendors].tolist()
        self.cwes = cwe[cwe["id"].isin(train.index)]["cwe_id"].value_counts().index[:top_cwes].tolist()
        self.tfidf = TfidfVectorizer(min_df=5, max_features=50000, ngram_range=(1, 2), sublinear_tf=True,
                                     stop_words="english")
        text = self.tfidf.fit_transform(train["text"])
        self.svd = TruncatedSVD(n_components=min(svd_dims, text.shape[1] - 1), random_state=RANDOM_STATE).fit(text)

    def transform(self, part):
        columns = {}
        for c in CATEGORICAL:
            codes = {value: i for i, value in enumerate(self.categories[c])}
            columns[c] = part[c].map(codes).astype(float)
        for c in NUMERIC:
            columns[c] = part[c].astype(float)
        vendor_flags = products[products["id"].isin(part.index) & products["vendor"].isin(self.vendors)]
        for i, vendor in enumerate(self.vendors):
            ids = vendor_flags.loc[vendor_flags["vendor"] == vendor, "id"].unique()
            columns[f"vendor_{i}"] = part.index.isin(ids).astype(float)
        cwe_flags = cwe[cwe["id"].isin(part.index) & cwe["cwe_id"].isin(self.cwes)]
        for i, cwe_id in enumerate(self.cwes):
            ids = cwe_flags.loc[cwe_flags["cwe_id"] == cwe_id, "id"].unique()
            columns[f"cwe_{i}"] = part.index.isin(ids).astype(float)
        text = self.svd.transform(self.tfidf.transform(part["text"]))
        for i in range(text.shape[1]):
            columns[f"text_{i}"] = text[:, i]
        return pd.DataFrame(columns, index=part.index)

    @property
    def categorical_mask(self):
        return [True] * len(CATEGORICAL) + [False] * (len(NUMERIC) + len(self.vendors) + len(self.cwes)
                                                     + self.svd.n_components)

def sample_weights(labels):
    positive_weight = (labels == 0).sum() / max((labels == 1).sum(), 1)
    return np.where(labels == 1, positive_weight, 1.0)

def fit_model(features, part, settings):
    model = HistGradientBoostingClassifier(categorical_features=features.categorical_mask, early_stopping=False,
                                           random_state=RANDOM_STATE, **settings)
    return model.fit(features.transform(part), part["label"], sample_weight=sample_weights(part["label"].values))

train = cves[cves["part"] == "1 train"]
validation = cves[cves["part"] == "2 validation"]
test = cves[cves["part"] == "4 test"].copy()

features = Features(train)
SETTINGS = [
    {"learning_rate": 0.05, "max_iter": 200, "max_leaf_nodes": 15, "l2_regularization": 1.0},
    {"learning_rate": 0.05, "max_iter": 400, "max_leaf_nodes": 31, "l2_regularization": 1.0},
    {"learning_rate": 0.1, "max_iter": 200, "max_leaf_nodes": 15, "min_samples_leaf": 50, "l2_regularization": 1.0},
]
validation_scores = []
for i, settings in enumerate(SETTINGS):
    model = fit_model(features, train, settings)
    p = model.predict_proba(features.transform(validation))[:, 1]
    validation_scores.append({"settings": i, "validation PR-AUC": average_precision_score(validation["label"], p)})
validation_scores = pd.DataFrame(validation_scores)
best = int(validation_scores.loc[validation_scores["validation PR-AUC"].idxmax(), "settings"])
print("Validation base rate:", round(validation["label"].mean(), 4), "| chosen settings:", SETTINGS[best])
validation_scores

Validation base rate: 0.004 | chosen settings: {'learning_rate': 0.05, 'max_iter': 400, 'max_leaf_nodes': 31, 'l2_regularization': 1.0}


,settings,validation PR-AUC
0,0,0.074312
1,1,0.083491
2,2,0.069196


In [6]:
known = cves[cves["part"].isin(["1 train", "2 validation"])]
final_features = Features(known)
final_model = fit_model(final_features, known, SETTINGS[best])
test["m2"] = final_model.predict_proba(final_features.transform(test))[:, 1]
print("Test:", len(test), "CVEs,", int(test["label"].sum()), "exploited within 180 days")

Test: 50341 CVEs, 154 exploited within 180 days


## 4. EPSS of the same date

For each test CVE, the EPSS score of the **Sunday after its publication**, from the dated daily files of EPSS (one file per Sunday). The files go to `data/staging/epss/`, which Git ignores. A CVE missing from that file (not yet scored) has no EPSS value: the comparison uses the CVEs that have one.

In [7]:
EPSS_DIR = ROOT / "data" / "staging" / "epss"
EPSS_DIR.mkdir(parents=True, exist_ok=True)
weekday = test["published"].dt.weekday  # Monday 0 ... Sunday 6
test["epss_day"] = test["published"] + pd.to_timedelta(((6 - weekday - 1) % 7) + 1, unit="D")

missing_files = []
for day in sorted(test["epss_day"].unique()):
    day = pd.Timestamp(day)
    path = EPSS_DIR / f"epss_scores-{day:%Y-%m-%d}.csv.gz"
    if not path.exists():
        url = f"https://epss.empiricalsecurity.com/epss_scores-{day:%Y-%m-%d}.csv.gz"
        try:
            urllib.request.urlretrieve(url, path)
        except Exception as error:
            missing_files.append((f"{day:%Y-%m-%d}", str(error)))
            continue
print("EPSS files:", test["epss_day"].nunique(), "Sundays,", len(missing_files), "missing", missing_files)

epss_parts = []
for day, group in test.groupby("epss_day"):
    path = EPSS_DIR / f"epss_scores-{pd.Timestamp(day):%Y-%m-%d}.csv.gz"
    if path.exists():
        scores = pd.read_csv(path, comment="#", usecols=["cve", "epss"])
        epss_parts.append(group[["cve_id"]].reset_index().merge(scores, left_on="cve_id", right_on="cve", how="inner"))
epss = pd.concat(epss_parts).set_index("id")["epss"] if epss_parts else pd.Series(dtype=float)
test["epss_at_prediction"] = epss.reindex(test.index)
print("Test CVEs with an EPSS score on the Sunday after publication:", f"{test['epss_at_prediction'].notna().mean():.1%}")

EPSS files: 48 Sundays, 0 missing []
Test CVEs with an EPSS score on the Sunday after publication: 100.0%


## 5. Evaluation (criterion 2)

Same CVEs for every model. Precision@k = share of exploited CVEs among the k CVEs with the highest score (ties broken at random, fixed seed). With about 0.3 % positives, a random ranking has a PR-AUC of about 0.003.

In [8]:
rng = np.random.RandomState(RANDOM_STATE)

def precision_at(labels, scores, k):
    order = np.lexsort((rng.random(len(scores)), -np.asarray(scores)))
    return labels[order[:k]].mean()

def evaluate(name, labels, scores):
    labels, scores = np.asarray(labels), np.asarray(scores, dtype=float)
    row = {"model": name, "CVEs": len(labels), "positives": int(labels.sum()),
           "PR-AUC": average_precision_score(labels, scores), "ROC-AUC": roc_auc_score(labels, scores)}
    for k in (50, 100, 500):
        row[f"precision@{k}"] = precision_at(labels, scores, k)
    return row

compared = test[test["epss_at_prediction"].notna()]
results = pd.DataFrame([
    evaluate("M2, all test CVEs", test["label"], test["m2"]),
    evaluate("M2", compared["label"], compared["m2"]),
    evaluate("EPSS (Sunday after publication)", compared["label"], compared["epss_at_prediction"]),
    evaluate("CNA CVSS score alone", compared["label"], compared["cna_score"].fillna(0)),
    evaluate("Random", compared["label"], rng.random(len(compared))),
]).round(4)
results

,model,CVEs,positives,PR-AUC,ROC-AUC,precision@50,precision@100,precision@500
0,"M2, all test CVEs",50341,154,0.0344,0.8746,0.08,0.09,0.058
1,M2,50339,154,0.0344,0.8746,0.08,0.09,0.058
2,EPSS (Sunday after publication),50339,154,0.1660,0.8452,0.50,0.35,0.114
3,CNA CVSS score alone,50339,154,0.0156,0.7430,0.04,0.05,0.040
4,Random,50339,154,0.0027,0.4456,0.00,0.00,0.004


In [9]:
def bootstrap_pr_auc(labels, scores, rounds=200):
    labels, scores = np.asarray(labels), np.asarray(scores, dtype=float)
    values = []
    for _ in range(rounds):
        sample = rng.randint(0, len(labels), len(labels))
        if labels[sample].sum() > 0:
            values.append(average_precision_score(labels[sample], scores[sample]))
    return np.percentile(values, [2.5, 97.5]).round(4)

print("PR-AUC 95 % interval (bootstrap, same CVEs):")
print("  M2  ", bootstrap_pr_auc(compared["label"], compared["m2"]))
print("  EPSS", bootstrap_pr_auc(compared["label"], compared["epss_at_prediction"]))

PR-AUC 95 % interval (bootstrap, same CVEs):
  M2   [0.0248 0.0539]
  EPSS [0.1158 0.2322]


## 6. Limits

- **Few positives**: about 150 exploited CVEs in the test part, so the metrics move a lot from one sample to another: see the bootstrap intervals above.
- **Latest version of the record**: the description and the affected products are those of today's record, not of the publication day. Exploitation phrases are masked, but a later rewording cannot be excluded.
- **Label horizon**: 180 days capture about 88 % of the eventual KEV additions (measured on the CVEs published since 2023); a CVE added later is labelled 0. KEV is also conservative: an exploited CVE that CISA never lists is labelled 0.
- **EPSS knows more**: the EPSS score of the Sunday after publication already uses up to 7 days of exploitation activity; M2 uses the record only. EPSS is expected to win: M2 is an honest benchmark (risk R4), not a replacement, and EPSS stays an input of the priority score.
- The comparison leaves out the test CVEs with no EPSS score on that Sunday (the share is printed in part 4).

## 7. Conclusion

Test period: CVEs published from 1 Jul 2025 to 24 May 2026, 50,341 CVEs, 154 exploited within 180 days (0.31 %). Same 50,339 CVEs for every model (2 have no EPSS score on the Sunday after publication).

| Model | PR-AUC [95 % interval] | ROC-AUC | precision@50 | precision@100 | precision@500 |
|---|---|---|---|---|---|
| EPSS, Sunday after publication | 0.166 [0.116 – 0.232] | 0.845 | 0.50 | 0.35 | 0.114 |
| **M2** | 0.034 [0.025 – 0.054] | 0.875 | 0.08 | 0.09 | 0.058 |
| CNA CVSS score alone | 0.016 | 0.743 | 0.04 | 0.05 | 0.040 |
| Random | 0.003 | 0.446 | 0.00 | 0.00 | 0.004 |

- M2 learns something real from the CNA record alone: its PR-AUC is about 13 times the random level and twice the CNA CVSS score.
- EPSS is clearly better where prioritization matters, at the top of the list: 25 of its 50 highest-scored CVEs were exploited, against 4 for M2, and the PR-AUC intervals do not overlap. EPSS already knows the first days of exploitation activity; M2 only knows the record.
- M2's ROC-AUC is higher than EPSS's: M2 separates the mass of unlikely CVEs well, but does not find the few exploited ones at the very top. ROC-AUC is dominated by the 50,000 negatives; PR-AUC and precision@k are the measures for ranking alerts.
- The validation PR-AUC was 0.083 (second half of 2024, 0.40 % positives) against 0.034 on the test (0.31 %): the newer period, with many more CVEs per quarter, is harder. Like M1, M2 would need regular retraining.
- Leakage check: features from the CNA record only, exploitation phrases counted and masked (part 2), time split with a gap so that no label unknown on 1 Jul 2025 is used (part 1).
- Decision for criterion 4: EPSS stays an input of the priority score; M2 is an honest benchmark, not a replacement. A possible use is a first estimate for a CVE that EPSS has not scored yet.